# Chapter 3 Practical 03: User-User kNN Prediction

Learning objectives:
- Select nearest neighbors for a target user.
- Predict missing ratings with a mean-centered, similarity-weighted formula.
- Explain a prediction through neighbor contributions.
- Generate Top-N user-user CF recommendations.

Slide connection: user-user CF, making predictions, prediction formula, and explainable memory-based CF.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


First, find users who have rated at least two of the same movies as the target user. Pearson is calculated only on those co-rated movies.


In [ ]:
def pearson_on_overlap(matrix, user_a, user_b):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    if pair.shape[1] < 2:
        return np.nan
    if pair.loc[user_a].std() == 0 or pair.loc[user_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair.loc[user_a], pair.loc[user_b])[0, 1])

def user_neighbors(matrix, target_user, min_overlap=2):
    rows = []
    for other in matrix.index.drop(target_user):
        overlap = matrix.loc[[target_user, other]].notna().all(axis=0).sum()
        sim = pearson_on_overlap(matrix, target_user, other)
        if overlap >= min_overlap and not pd.isna(sim):
            rows.append({"neighbor": other, "similarity": sim, "overlap": int(overlap)})
    return pd.DataFrame(rows).sort_values("similarity", ascending=False)

user_neighbors(rating_matrix, "Karen")


The prediction starts from the target user's average rating. Each neighbor then contributes how much their rating for the target item is above or below their own average.


In [ ]:
def predict_user_user(matrix, target_user, target_item, k_neighbors=3, min_overlap=2, positive_only=True):
    target_mean = matrix.loc[target_user].mean()
    neighbors = user_neighbors(matrix, target_user, min_overlap=min_overlap)
    neighbors = neighbors[neighbors["neighbor"].map(lambda u: not pd.isna(matrix.loc[u, target_item]))]
    if positive_only:
        # In this introductory notebook, negative similarities are excluded because they indicate opposite taste.
        neighbors = neighbors[neighbors["similarity"] > 0]
    neighbors = neighbors.head(k_neighbors)
    if neighbors.empty:
        return np.nan, neighbors

    numerator = 0.0
    denominator = 0.0
    rows = []
    for _, row in neighbors.iterrows():
        u = row["neighbor"]
        sim = row["similarity"]
        neighbor_mean = matrix.loc[u].mean()
        centered_rating = matrix.loc[u, target_item] - neighbor_mean
        contribution = sim * centered_rating
        numerator += contribution
        denominator += abs(sim)
        rows.append({
            "neighbor": u,
            "similarity": sim,
            "neighbor_rating": matrix.loc[u, target_item],
            "neighbor_mean": neighbor_mean,
            "centered_rating": centered_rating,
            "weighted_contribution": contribution,
        })
    prediction = target_mean + numerator / denominator if denominator else np.nan
    return prediction, pd.DataFrame(rows)

target_user = "Karen"
target_item = "Independence Day"
k_neighbors = 3

pred, evidence = predict_user_user(rating_matrix, target_user, target_item, k_neighbors=k_neighbors)
print(f"Predicted Karen rating for Independence Day: {pred:.2f}")
evidence.round(3)


Now repeat the same prediction for every movie the target user has not rated, then rank the predicted ratings.


In [ ]:
def recommend_user_user(matrix, target_user, n=5, k_neighbors=3):
    unseen_items = matrix.columns[matrix.loc[target_user].isna()]
    rows = []
    for item in unseen_items:
        pred, evidence = predict_user_user(matrix, target_user, item, k_neighbors=k_neighbors)
        if not pd.isna(pred):
            rows.append({
                "user": target_user,
                "recommended_movie": item,
                "predicted_rating": pred,
                "supporting_neighbors": ", ".join(evidence["neighbor"].tolist()),
            })
    return pd.DataFrame(rows).sort_values("predicted_rating", ascending=False).head(n)

recommend_user_user(rating_matrix, target_user, n=5, k_neighbors=k_neighbors).round(2)


# Challenges

### Challenge 1 — Change the Neighborhood Size

**Goal:**
Investigate how the number of nearest neighbors changes a user-user prediction.

**What to do:**

1. Change the existing `k_neighbors` value from `3` to `1`.
2. Rerun the prediction and recommendation cells.
3. Change `k_neighbors` again to `5`.
4. Compare the predicted rating and supporting neighbors.


In [ ]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

How did the prediction change when `k_neighbors` changed? Which value gave the clearest explanation? Why can too few or too many neighbors be risky?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Change the Target User and Item

**Goal:**
Investigate whether the same user-user CF method behaves differently for another prediction target.

**What to do:**

1. Change `target_user` from `"Karen"` to `"Alice"`.
2. Change `target_item` to an unseen movie for Alice, such as `"Toy Story"`.
3. Rerun the neighbor, prediction, and recommendation cells.
4. Compare the new evidence table with the original one.


In [ ]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Did the selected neighbors change? Was there enough neighbor evidence for the new target item? How did the explanation differ from Karen's prediction?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Negative Similarity

This challenge requires **no programming**.

In the notebook, negative similarities are excluded from the beginner prediction.

Explain what a negative user-user similarity means and why including it without explanation could confuse a simple rating prediction.

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
